# Gold Customer Dimension

## Objective
Build a business-ready customer dimension from the Silver Customers dataset.

## Responsibilities
- Load the accepted Silver customer data.
- Select the customer attributes required for analytics.
- Add Gold processing metadata.
- Validate the final schema, row count, and key integrity.
- Write the Gold dimension to Parquet and verify the saved output.

**Grain:** One row per customer.
**Primary key:** `customer_id`.

### Configuration / Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"

SILVER_CUSTOMER_PATH = f"{FILE_PATH}/silver/customers"
GOLD_CUSTOMER_PATH = f"{FILE_PATH}/gold/dim_customer"

EXPECTED_CUSTOMER_COUNT = 9997

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("gold-dim-customer")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


### Load Data

In [ ]:
df_silver_customer = spark.read.parquet(
    SILVER_CUSTOMER_PATH
)


silver_customer_count = df_silver_customer.count()

print("Silver customer count:", silver_customer_count)

Silver customer count: 9997


### Inspect Source Data

In [ ]:
df_silver_customer.printSchema()

df_silver_customer.show(10, truncate=False)

print("Available columns:")
print(df_silver_customer.columns)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+-----------+----------+---------+---------------------------+--------------+--------+-----------+----------------+--------------+---------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_id|first_name|last_name|email                      |country       |city    |signup_date|customer_segment|account_status|_silver_processed_timestamp|_source_file                                      

### Select Gold Customer Attributes

In [ ]:
GOLD_CUSTOMER_COLUMNS = [
    "customer_id",
    "first_name",
    "last_name",
    "country",
    "city",
    "signup_date",
    "customer_segment",
    "account_status",
]

df_gold_customer = df_silver_customer.select(
    *GOLD_CUSTOMER_COLUMNS
)

print("Selected customer attributes:")
df_gold_customer.printSchema()

Selected customer attributes:
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)



### Add Gold Audit Metadata

In [ ]:
df_gold_customer = df_gold_customer.withColumn(
    "_gold_processed_timestamp",
    F.current_timestamp()
)

df_gold_customer.show(10, truncate=False)

+-----------+----------+---------+--------------+--------+-----------+----------------+--------------+--------------------------+
|customer_id|first_name|last_name|country       |city    |signup_date|customer_segment|account_status|_gold_processed_timestamp |
+-----------+----------+---------+--------------+--------+-----------+----------------+--------------+--------------------------+
|C000001    |Mia       |Davis    |Germany       |Munich  |2024-10-14 |Small Business  |Active        |2026-10-28 21:25:02.153234|
|C000002    |Arjun     |Davis    |Canada        |Calgary |2025-08-11 |Consumer        |Active        |2026-10-28 21:25:02.153234|
|C000003    |Noah      |Wilson   |Canada        |Calgary |2021-02-24 |Consumer        |Active        |2026-10-28 21:25:02.153234|
|C000004    |Liam      |Taylor   |United States |Denver  |2022-12-17 |Consumer        |Inactive      |2026-10-28 21:25:02.153234|
|C000005    |Sophia    |Smith    |Canada        |Calgary |2021-04-05 |Consumer        |Ina

### Validate Gold Customer Dimension

In [ ]:
gold_customer_count = df_gold_customer.count()

expected_columns = GOLD_CUSTOMER_COLUMNS + [
    "_gold_processed_timestamp"
]

actual_columns = df_gold_customer.columns

assert actual_columns == expected_columns, (
    f"Unexpected schema. Expected {expected_columns}, "
    f"received {actual_columns}"
)

assert gold_customer_count == silver_customer_count, (
    f"Row count changed: Silver={silver_customer_count}, "
    f"Gold={gold_customer_count}"
)

null_customer_ids = df_gold_customer.filter(
    F.col("customer_id").isNull()
).count()

assert null_customer_ids == 0, (
    f"Found {null_customer_ids} null customer IDs"
)

null_gold_timestamps = df_gold_customer.filter(
    F.col("_gold_processed_timestamp").isNull()
).count()

assert null_gold_timestamps == 0, (
    f"Found {null_gold_timestamps} null Gold timestamps"
)

print("Gold customer validation passed.")
print("Gold customer count:", gold_customer_count)

Gold customer validation passed.
Gold customer count: 9997


### Write Gold Customer Dimension

In [ ]:
(
    df_gold_customer.write
    .format("parquet")
    .mode("overwrite")
    .save(GOLD_CUSTOMER_PATH)
)

print("Gold customer dimension written to:")
print(GOLD_CUSTOMER_PATH)

Gold customer dimension written to:
/content/drive/MyDrive/pyspark-ecommerce-analytics-data/gold/dim_customer


### Read Back and Verify

In [ ]:
df_gold_customer_saved = spark.read.parquet(
    GOLD_CUSTOMER_PATH
)

saved_customer_count = df_gold_customer_saved.count()

assert saved_customer_count == gold_customer_count, (
    f"Saved count mismatch: expected {gold_customer_count}, "
    f"received {saved_customer_count}"
)

assert df_gold_customer_saved.columns == expected_columns, (
    "Saved Gold schema does not match the expected schema"
)

saved_null_customer_ids = df_gold_customer_saved.filter(
    F.col("customer_id").isNull()
).count()

assert saved_null_customer_ids == 0, (
    "Saved Gold dimension contains null customer IDs"
)

df_gold_customer_saved.printSchema()
df_gold_customer_saved.show(10, truncate=False)

print("Read-back verification passed.")

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- _gold_processed_timestamp: timestamp (nullable = true)

+-----------+----------+---------+--------------+--------+-----------+----------------+--------------+--------------------------+
|customer_id|first_name|last_name|country       |city    |signup_date|customer_segment|account_status|_gold_processed_timestamp |
+-----------+----------+---------+--------------+--------+-----------+----------------+--------------+--------------------------+
|C000001    |Mia       |Davis    |Germany       |Munich  |2024-10-14 |Small Business  |Active        |2026-10-28 21:32:25.283485|
|C000002    |Arjun     |Davis    |Canada        |Calgary |2025-08-11 |Consumer

### Quality Metrics and Run Summary

In [ ]:
summary = {
    "silver_customer_count": silver_customer_count,
    "gold_customer_count": gold_customer_count,
    "saved_customer_count": saved_customer_count,
    "null_customer_ids": saved_null_customer_ids,
    "columns_in_gold": len(expected_columns),
    "output_path": GOLD_CUSTOMER_PATH,
    "validation_status": "PASSED",
}

for metric, value in summary.items():
    print(f"{metric}: {value}")

silver_customer_count: 9997
gold_customer_count: 9997
saved_customer_count: 9997
null_customer_ids: 0
columns_in_gold: 9
output_path: /content/drive/MyDrive/pyspark-ecommerce-analytics-data/gold/dim_customer
validation_status: PASSED
